In [2]:
!pip install folktables

In [3]:
!pip install owlready2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 73.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for owlready2: filename=owlready2-0.51-cp313-cp313-linux_x86_64.whl size=24764653 sha256=a57da081735c060a87d324ff70ae9043eb0b626fe7970702f19d78629fc0fa63
  Stored in directory: /root/.cache/pip/wheels/cc/16/52/762fdfa8c53f256db44674790a3ceba702c3c763c96ce8b62c
Successfully built owlready2


# Imports

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from folktables import ACSDataSource, ACSIncome
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import owlready2 as or2

# Graph setup

In [ ]:
onto = or2.get_ontology("/path/to/ontology/mlprov.owx").load()
with onto:
    class description(or2.DataProperty):
        range = [str]

* Owlready2 * WARNING: ObjectProperty http://www.w3.org/ns/prov#wasRevisionOf belongs to more than one entity types: [owl.AnnotationProperty, owl.ObjectProperty, prov.wasDerivedFrom]; I'm trying to fix it...
* Owlready2 * WARNING: ObjectProperty http://www.w3.org/ns/prov#specializationOf belongs to more than one entity types: [owl.AnnotationProperty, owl.ObjectProperty, prov.alternateOf]; I'm trying to fix it...


In [7]:
req_spec = onto["requirement_specification"]
train_data = onto["training_dataset"]
test_data = onto["testing_dataset"]
selected_feature = onto["selected_feature"]
onto_model = onto["model"]
performance_metric = onto["performance_metric"]

In [8]:
req1 = req_spec("req1")
req1.description = ["The model shall predict the class <=50K."]
req2 = req_spec("req2")
req2.description = ["The model shall predict the class >50K."]
req3 = req_spec("req3")
req3.description = ["The model shall predict Alabama income patterns."]


# Load the data

In [9]:
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
ca_data = data_source.get_data(states=["CA"], download=True)
al_data = data_source.get_data(states=["AL"], download=True)

ca_features, ca_labels, _ = ACSIncome.df_to_numpy(ca_data)
al_features, al_labels, _ = ACSIncome.df_to_numpy(al_data)

training_dataset = train_data("ca_data")
training_dataset.description = ["ACSIncome CA data."]

testing_dataset = test_data("al_data")
testing_dataset.description = ["ACSIncome AL data."]

ca = selected_feature("State=CA")
al = selected_feature("State=AL")

# Mismatch

In [10]:
mismatched_model = HistGradientBoostingClassifier(max_iter=10000)
mismatched_model.fit(ca_features, ca_labels)      # train on CA


HistGradientBoostingClassifier(max_iter=10000)

In [11]:
mismatch_pred = mismatched_model.predict(al_features)

mismatch_acc = accuracy_score(al_labels, mismatch_pred)
print(mismatch_acc)
mismtached_accuracy = performance_metric(f"Mismatched_accuracy={mismatch_acc}")

mismatch_prec = precision_score(al_labels, mismatch_pred)
print(mismatch_prec)
mismtached_precision = performance_metric(f"Mismatched_precision={mismatch_prec}")

mismatch_rec = recall_score(al_labels, mismatch_pred)
print(mismatch_rec)
mismtached_recall = performance_metric(f"Mismatched_recall={mismatch_rec}")

mismatch_f1 = f1_score(al_labels, mismatch_pred)
print(mismatch_f1)
mismtached_f1_score = performance_metric(f"Mismatched_F1_score={mismatch_f1}")

0.7622148374348842
0.5806036615536863
0.8473425765453495
0.6890598390980093


# Matching

In [12]:
al_X_train, al_X_test, al_y_train, al_y_test = train_test_split(
    al_features, al_labels, test_size=0.2
)

model_al_only = HistGradientBoostingClassifier(max_iter=10000)
model_al_only.fit(al_X_train, al_y_train)

HistGradientBoostingClassifier(max_iter=10000)

In [13]:
matching_pred = model_al_only.predict(al_X_test)


matching_acc = accuracy_score(al_y_test, matching_pred)
print(matching_acc)
matching_accuracy = performance_metric(f"Al_only_accuracy={matching_acc}")

matching_prec = precision_score(al_y_test, matching_pred)
print(matching_prec)
matching_precision = performance_metric(f"Al_only_precision={matching_prec}")

matching_rec = recall_score(al_y_test, matching_pred)
print(matching_rec)
matching_recall = performance_metric(f"Al_only_recall={matching_rec}")

matching_f1 = f1_score(al_y_test, matching_pred)
print(matching_f1)
matching_f1_score = performance_metric(f"Al_only_F1_score={matching_f1}")

0.8190390660080826
0.7335423197492164
0.6676176890156919
0.6990291262135923


In [14]:
print(f"Percent change accuracy: {(mismatch_acc-matching_acc)/mismatch_acc*100}")
print(f"Percent change precision: {(mismatch_prec-matching_prec)/mismatch_prec*100}")
print(f"Percent change recall: {(mismatch_rec-matching_rec)/mismatch_rec*100}")
print(f"Percent change F1 score: {(mismatch_f1-matching_f1)/mismatch_f1*100}")

Percent change accuracy: -7.455145948671321
Percent change precision: -26.341318238722206
Percent change recall: 21.210416247747563
Percent change F1 score: -1.4467955538713362


In [ ]:
onto.save(file="/path/mlprov_eval_mismatch_bias_instances.owl")